# Can one model reconcile product records, equipment manuals, images, and campsite constraints into an evidence-backed outfitting plan?

## 1. Before You Begin

We are helping a Contoso Outdoors customer prepare for a rainy, three-season
family campsite. The evidence is intentionally mixed: two catalog records, two
manuals, and two product images. Our single focus is **multimodal evidence
synthesis**—combining those sources while keeping their claims distinct.

Deploy `gpt-6-astra` in Microsoft Foundry, set the variables shown in section
2, and review the [repository quickstart](../../quickstart/README.md). Current
rates vary by deployment and context length; use the
[Azure OpenAI pricing page](https://azure.microsoft.com/pricing/details/azure-openai/).


In [ ]:
# 2. Verify your environment
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()


def find_assets() -> Path:
    """Find the shared data whether Jupyter starts here or at the repo root."""
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / "models/azure-openai/shared/contoso-outdoors"
        if (candidate / "products.json").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find models/azure-openai/shared/contoso-outdoors. "
        "Run this notebook from a checkout of the model-releases repository."
    )


required = ["AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_API_KEY", "AZURE_OPENAI_GPT_6_ASTRA_DEPLOYMENT"]
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise EnvironmentError(
        f"Missing {missing}. Copy scripts/sample.env to .env, add the values, "
        "and review models/quickstart/README.md."
    )

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
base_url = endpoint if endpoint.endswith("/openai/v1") else f"{endpoint}/openai/v1"
client = OpenAI(api_key=os.environ["AZURE_OPENAI_API_KEY"], base_url=f"{base_url}/")
deployment = os.environ["AZURE_OPENAI_GPT_6_ASTRA_DEPLOYMENT"]
assets = find_assets()

expected_assets = [
    assets / "products.json",
    assets / "manuals/product_info_1.md",
    assets / "manuals/product_info_2.md",
    assets / "images/product_1.webp",
    assets / "images/product_2.webp",
]
missing_assets = [str(path) for path in expected_assets if not path.is_file()]
if missing_assets:
    raise FileNotFoundError(f"Shared Contoso Outdoors files are missing: {missing_assets}")

print(f"Environment ready for deployment: {deployment}")
print(f"Shared assets: {assets}")


## 3. Inspect the shared evidence

The product records give us concise merchandising facts. The manuals add setup,
care, and safety detail. The images can support visual observations, but they
must not override written specifications. We load all five local files before
making a network request so a missing fixture fails early.


In [ ]:
# 4. Load the local product evidence
import base64
import json

from IPython.display import HTML, display

# These are the only product records used in this phase.
products = json.loads((assets / "products.json").read_text(encoding="utf-8"))
manuals = {
    product["id"]: (assets / product["manual"]).read_text(encoding="utf-8")
    for product in products
}

def display_webp(path: Path, width: int = 240) -> None:
    """Render a local WebP through HTML because IPython Image cannot embed it."""
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    display(HTML(f'<img src="data:image/webp;base64,{encoded}" width="{width}">'))


for product in products:
    print(f'{product["id"]}: {product["name"]} (${product["price"]})')
    display_webp(assets / product["images"][0])

assert [product["id"] for product in products] == [1, 2]


## 5. Ask for an evidence-backed plan

We send text and images in one Responses API request. The prompt requires every
claim to name its source and asks the model to label uncertainty rather than
silently resolving contradictions.


In [ ]:
# 6. Synthesize text and image evidence with the Responses API
def image_data_url(path: Path) -> str:
    """Encode a local WebP image for an input_image content item."""
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:image/webp;base64,{encoded}"


catalog_text = json.dumps(products, indent=2)
manual_text = "\n\n".join(
    f"MANUAL FOR PRODUCT ID {product_id}\n{manual}"
    for product_id, manual in manuals.items()
)
task = f"""
You are reviewing evidence for a rainy, three-season family campsite.
Recommend which of these two products belong in the customer's outfitting
plan. This is not a request to invent a complete kit.

For each recommendation:
1. cite the product ID;
2. identify whether each claim came from CATALOG, MANUAL, or IMAGE;
3. call out contradictory or missing information;
4. include a short safety/setup checklist grounded only in the supplied text.

CATALOG
{catalog_text}

MANUALS
{manual_text}
"""

content = [{"type": "input_text", "text": task}]
for product in products:
    image_path = assets / product["images"][0]
    content.append({"type": "input_image", "image_url": image_data_url(image_path)})

response = client.responses.create(
    model=deployment,
    input=[{"role": "user", "content": content}],
    reasoning={"effort": "high"},
)
answer = response.output_text
print(answer)

# This check confirms coverage, not factual correctness; a developer still reviews the citations.
for product in products:
    assert str(product["id"]) in answer, f"Response did not cite product ID {product['id']}"
print(f"\nUsage: {response.usage}")


## 7. Your Turn to Explore

Try one direction without changing the shared source files:

- Remove one image and note which visual claims disappear.
- Ask for a compact table with separate `CATALOG`, `MANUAL`, and `IMAGE` columns.
- Lower reasoning effort and compare evidence coverage, not writing style.


## 8. Summary

We used GPT-6 Astra for one task: synthesizing local text and image evidence
without blurring source boundaries. Reach for this pattern when a decision
depends on mixed evidence and explicit uncertainty matters. For background,
review the [multimodal](../../../docs/primers/multimodal-models.md) and
[reasoning](../../../docs/primers/reasoning-models.md) primers.


## 9. References

- [GPT-6 Astra model card](https://ai.azure.com/catalog/models/gpt-6-astra) — model and deployment overview.
- [Azure OpenAI reasoning models](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/reasoning) — reasoning controls and usage.
- [Use the Azure OpenAI Responses API](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/responses) — Python and multimodal request patterns.
- [Foundry Models sold by Azure](https://learn.microsoft.com/en-us/azure/foundry/foundry-models/concepts/models-sold-directly-by-azure#gpt-6) — supported version and limits.
